# 🏋️‍♂️ CMAT-HOI: Cross-Modal Attention Transformer for Sports Action Recognition
## 🚀 Google Colab Cloud Master Notebook (82 Sports Categories)

**Thesis Framework**: *Cross-Modal Attention Transformer for Human–Object Interaction (CMAT-HOI)*  
**Performance**: **90.19% Top-1 Accuracy** | **99.02% Top-5 Accuracy** across 82 categories (9,151 videos)

---

### 📌 What You Can Do in this Notebook:
1. **Option 1 (Fastest - No Training Needed)**: Load the pre-trained `best_model.pth` and test custom videos or live webcams instantly.
2. **Option 2 (Train from Scratch)**: Train the full CMAT-HOI Transformer on Colab's cloud GPU (T4 / V100 / A100) and generate all 7 thesis publication diagrams.

## 1. Verify Colab GPU Acceleration
Make sure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU`).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Ready: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model  : {torch.cuda.get_device_name(0)}")

## 2. Clone GitHub Codebase & Install Requirements

In [ ]:
# Clone fresh codebase from GitHub
!rm -rf Sport-Action-Detection
!git clone https://github.com/angkon101/Sport-Action-Detection.git
%cd Sport-Action-Detection

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q gradio

---
## ⚡ OPTION 1: Load Pre-Trained Model & Run Inference (Instant)
If you already have `best_model.pth` (from your local machine `checkpoints/M7/best_model.pth` or Google Drive), run this section to test videos without training!

### Step 3A: Load `best_model.pth` from Google Drive (or direct upload)
Mount your Google Drive to load `best_model.pth` (or upload it directly via the file picker below):

In [ ]:
from pathlib import Path
import shutil

target_dir = Path("checkpoints/M7")
target_dir.mkdir(parents=True, exist_ok=True)
target_ckpt = target_dir / "best_model.pth"

# Check if Google Drive has best_model.pth
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_candidates = [
        Path('/content/drive/MyDrive/best_model.pth'),
        Path('/content/drive/MyDrive/checkpoints/M7/best_model.pth'),
        Path('/content/drive/MyDrive/Sports_Action_Results/checkpoints/M7/best_model.pth'),
    ]
    for cand in drive_candidates:
        if cand.exists():
            shutil.copy2(cand, target_ckpt)
            print(f"[✓] Found and copied trained weights from Google Drive: {cand}")
            break
except Exception as e:
    print(f"[!] Google Drive mount info: {e}")

# Fallback: If not in Drive, prompt direct browser upload
if not target_ckpt.exists():
    print("\n[i] 'best_model.pth' not found in Google Drive root.")
    print("    Please upload your 'best_model.pth' file (from your local 'checkpoints/M7/' folder):")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(('.pth', '.pt')):
            shutil.move(fname, str(target_ckpt))
            print(f"[✓] Model checkpoint saved to: {target_ckpt}")
            break

if target_ckpt.exists():
    print(f"\n[SUCCESS] Model weights ready at: {target_ckpt} ({target_ckpt.stat().st_size / (1024*1024):.1f} MB)")

### Step 3B: Test the Model on Any Video File

In [ ]:
from src.inference import ActionInferenceEngine
from google.colab import files

# 1. Initialize CMAT-HOI Inference Engine
engine = ActionInferenceEngine(checkpoint_path="checkpoints/M7/best_model.pth")
print("\n[✓] CMAT-HOI Model successfully loaded on Colab GPU!")

# 2. Upload any video to test (.mp4, .avi, .mov)
print("Upload a sports video to classify:")
uploaded_video = files.upload()
video_path = list(uploaded_video.keys())[0]

# 3. Run multimodal prediction
print(f"\nAnalyzing video: {video_path}...")
result = engine.predict_video(video_path)

print("=" * 60)
print(f"PREDICTED SPORT ACTION : {result['top_class'].upper()}")
print(f"CONFIDENCE             : {result['top_confidence']:.2f}%")
print("=" * 60)
print("Top-5 Action Rankings:")
for idx, (cls_name, prob) in enumerate(result['predictions'], start=1):
    print(f"  {idx}. {cls_name:<25}: {prob:.2f}%")

### Step 3C: Launch Interactive Gradio Web App (Webcam & Share Link)
Runs a live web dashboard with a public URL (`https://xxxx.gradio.live`) to test videos or use your webcam directly from your browser or mobile phone!

In [ ]:
!python app_web.py --share

---
## 🏋️‍♂️ OPTION 2: Train CMAT-HOI from Scratch on Colab
Use this section if you want to train the model on Colab's GPU using `Model Data.zip`.

### Step 4A: Load Dataset (`Model Data.zip`)
Mount your Google Drive to load `Model Data.zip`:

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive')

if (drive_path / 'Model Data.zip').exists():
    print("Found 'Model Data.zip' in Google Drive. Extracting...")
    !unzip -q -o "/content/drive/MyDrive/Model Data.zip" -d ./
elif (drive_path / 'ucf action.zip').exists():
    print("Found 'ucf action.zip' in Google Drive. Extracting...")
    !unzip -q -o "/content/drive/MyDrive/ucf action.zip" -d ./
    !python data/scripts/populate_model_data_split.py --source-dir "ucf action" --target-dir "Model Data"
else:
    print("[!] Please upload 'Model Data.zip' to your Google Drive root and rerun this cell.")

### Step 4B: Extract Features & Verify Integrity

In [ ]:
# Extract multimodal features into data/features
!python run_local.py --extract --data-dir "Model Data" --features-dir "data/features"

# Pre-flight integrity verification
!python run_local.py --verify --features-dir "data/features"

### Step 4C: Train CMAT-HOI (Cross-Modal Attention Transformer)
Trains CMAT-HOI for 45 epochs with AdamW, Cosine Annealing, and Focal Loss with Label Smoothing.

In [ ]:
!python run_local.py --train --model-mode M7 --arch transformer --epochs 45 --batch-size 64 --lr 0.0004

### Step 4D: View All 7 Publication Thesis Diagrams

In [ ]:
from IPython.display import Image, display
from pathlib import Path

diag_dir = Path("diagrams/M7")
diagrams = [
    ("Diagram 1: System Architecture Diagram", diag_dir / "system_architecture_diagram.png"),
    ("Diagram 2: Training & Validation Loss Curves", diag_dir / "loss_curves.png"),
    ("Diagram 3: Top-1 (90.19%) & Top-5 (99.02%) Accuracy Curves", diag_dir / "accuracy_curves.png"),
    ("Diagram 4: 82-Class Normalized Confusion Matrix", diag_dir / "confusion_matrix_82.png"),
    ("Diagram 5: Per-Class Accuracy Rankings (Top 10 vs Bottom 10)", diag_dir / "per_class_accuracy.png"),
    ("Diagram 6: Temporal Attention Allocation Across 16 Frames", diag_dir / "temporal_attention_weights.png"),
    ("Diagram 7: Metric Distribution (Precision, Recall, F1)", diag_dir / "metric_distribution_82.png"),
    ("Diagram 8: Multimodal Stream Ablation Study (M1 to M7)", diag_dir / "multimodal_ablation_comparison.png"),
]

for title, img_path in diagrams:
    if img_path.exists():
        print(f"\n{'='*75}\n{title}\n{'='*75}")
        display(Image(filename=str(img_path), width=850))
    else:
        print(f"[!] Diagram not found: {img_path}")

### Step 4E: Backup Checkpoints & Diagrams to Google Drive

In [ ]:
!mkdir -p "/content/drive/MyDrive/Sports_Action_Results"
!cp -r diagrams/ "/content/drive/MyDrive/Sports_Action_Results/"
!cp -r checkpoints/ "/content/drive/MyDrive/Sports_Action_Results/"
!zip -r /content/Sports_Action_Results.zip diagrams checkpoints

print("\n[✓] Results permanently saved to Google Drive: /content/drive/MyDrive/Sports_Action_Results/")
print("[✓] Downloadable archive ready: /content/Sports_Action_Results.zip")